# Modelado — ¿sobrevivió el pasajero?

Pregunta predictiva: dadas las características de un pasajero conocidas al embarcar, ¿con qué probabilidad habría sobrevivido y qué tan confiable es esa estimación?

La lógica reutilizable está en `flows/05_ml_models.py`; el cuaderno la usa y analiza los resultados.

## 10. Formulación del problema

In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    brier_score_loss,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    cross_validate,
    learning_curve,
)

from flows.common.paths import BRONZE, BRONZE_UNLABELED_NAME, MODELS, SILVER, SILVER_NAME

# Los módulos con nombre numérico se importan con importlib
ml = importlib.import_module("flows.05_ml_models")
de = importlib.import_module("flows.01_data_engineering")

sns.set_theme(style='whitegrid', context='notebook')
X, y = ml.load_dataset()          # lee Silver; X con columnas candidatas, y = Survived
print(X.shape, y.mean())

### Paso 39. Problema y métrica

La definición del problema, la métrica y el criterio de éxito están en `reports/problema_ml.md`.

### Paso 40. Auditoría de variables y fuga de datos

| Columna | ¿Existe antes del evento? | ¿Riesgo de fuga? | Decisión |
|---|---|---|---|
| `Survived`, `Survivor` | No (es el objetivo) | Fuga directa | Excluir |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores: memorizan, no generalizan | Excluir (el título entra como `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican a `Sex`, etc. con un orden arbitrario | Excluir (se usa *one-hot*) |
| `Age` | — | Está imputada con estadísticas de todos los datos | Usar `AgeOriginal` |
| `AgeImputada` | — | Se deriva de la imputación | Excluir |
| `Cabin`, `TCabin`, `HasCabin` | Dudoso | La cabina pudo registrarse después del naufragio | Evaluar con y sin ella (Paso 61) |
| `Person` | Derivada de `Age` y `Sex` | Redundante con ambas | Excluir del modelo base |
| `Pclass`, `Sex`, `SibSp`, `Parch`, `Fare`, `Embarked` | Sí | Sin riesgo | Incluir |
| `TTitulo`, `Familia`, `Alone` | Sí (salen de `Name`, `SibSp`, `Parch`) | Sin riesgo | Incluir |

In [ ]:
silver = pd.read_parquet(SILVER / SILVER_NAME)
print(silver.columns.tolist())
print(ml.CANDIDATE_COLUMNS)

### Paso 41. División entrenamiento / prueba

In [ ]:
X_train, X_test, y_train, y_test = ml.split(X, y)
print(X_train.shape, X_test.shape)
print('Supervivencia en entrenamiento:', round(y_train.mean(), 3))
print('Supervivencia en prueba:', round(y_test.mean(), 3))

**Resultado:** 712 pasajeros para entrenamiento y 179 para prueba. La tasa de supervivencia es casi igual en ambos (38.3 % y 38.5 %), así que la estratificación funcionó. `X_test` no se vuelve a usar hasta el Paso 62.

### Paso 42. Baselines

Todo se evalúa con validación cruzada sobre `X_train` (los mismos pliegues para todos).

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=ml.settings.random_state)

# Baseline ingenuo: siempre la clase más frecuente
dummy = ml.build_pipeline(DummyClassifier(strategy='most_frequent'))
res_dummy = cross_validate(dummy, X_train, y_train, cv=cv,
                           scoring=['accuracy', 'precision', 'recall', 'f1'])
print('Dummy:', {k[5:]: round(v.mean(), 3) for k, v in res_dummy.items() if k.startswith('test_')})

# Baseline regla de género: predice 1 si Sex == 'female'
filas = []
for _, idx in cv.split(X_train, y_train):
    real = y_train.iloc[idx]
    pred = (X_train.iloc[idx]['Sex'] == 'female').astype(int)
    filas.append({'accuracy': accuracy_score(real, pred), 'precision': precision_score(real, pred),
                  'recall': recall_score(real, pred), 'f1': f1_score(real, pred),
                  'roc_auc': roc_auc_score(real, pred)})
regla = pd.DataFrame(filas).mean().round(3)
print('Regla de género:', regla.to_dict())

**Resultado:** el dummy acierta 61.7 % (siempre dice "no sobrevive") con recall 0. La regla de género llega a exactitud 0.789 y F1 0.714. **La vara a superar es ≈ 0.79 de exactitud.**

## 11. Preprocesamiento con `Pipeline`

### Paso 43. Demostrar la fuga de datos

In [ ]:
mujeres_todos = X.loc[X['Sex'] == 'female', 'Age'].mean()
mujeres_train = X_train.loc[X_train['Sex'] == 'female', 'Age'].mean()
print('Edad media de mujeres con TODOS los datos:', round(mujeres_todos, 4))
print('Edad media de mujeres solo con X_train:', round(mujeres_train, 4))
print('Diferencia:', round(mujeres_todos - mujeres_train, 4))

**Resultado:** la edad media de las mujeres es 27.92 con todos los datos y 28.18 solo con entrenamiento (diferencia de 0.27 años). En el primer caso, las edades de las mujeres de `X_test` influyen en el promedio que se usa para entrenar. La diferencia es pequeña porque hay muchas mujeres con edad conocida y el promedio casi no cambia. Aun así es un error de método: con más faltantes o grupos pequeños la diferencia podría ser grande.

### Paso 46. Verificar el preprocesamiento

In [ ]:
pipe = ml.build_pipeline(DummyClassifier())
pre = pipe[:-1].fit(X_train)
Z = pre.transform(X_test)
print('¿Hay NaN?', np.isnan(Z).any())
print('Columnas:', Z.shape[1])
pre['prep'].get_feature_names_out()

**Resultado:** después del preprocesamiento no quedan `NaN` y salen 19 columnas. `Sex` queda como `cat__Sex_female` y `cat__Sex_male`, sin un orden numérico implícito.

**Decisión (Paso 45):** imputar la edad por `Sex` o por `Sex` y `Pclass`. Se compara con regresión logística y validación cruzada.

In [ ]:
for grupo in [('Sex',), ('Sex', 'Pclass')]:
    p = ml.build_pipeline(ml.get_candidates(42)['logreg']).set_params(age_imputer__group_cols=grupo)
    puntajes = cross_val_score(p, X_train, y_train, cv=cv, scoring='roc_auc')
    print(grupo, round(puntajes.mean(), 4), '+/-', round(puntajes.std(), 4))

**Resultado:** imputar por `Sex` (0.8584) o por `Sex` y `Pclass` (0.8585) da casi el mismo ROC-AUC. Se mantiene `Sex` y `Pclass` porque es más específica y no cuesta nada, aunque la diferencia no es real.

## 12. Modelos candidatos y validación cruzada

### Paso 48. Validación cruzada de todos los candidatos

In [ ]:
resultados = ml.cross_validate_candidates(X_train, y_train)
resultados.round(3)

### Paso 49. Leer los resultados

In [ ]:
filas = []
for nombre, modelo in ml.get_candidates(42).items():
    res = cross_validate(ml.build_pipeline(modelo), X_train, y_train, cv=cv, scoring='roc_auc')
    for pliegue, valor in enumerate(res['test_score']):
        filas.append({'modelo': nombre, 'pliegue': pliegue, 'roc_auc': valor})
por_pliegue = pd.DataFrame(filas)
sns.boxplot(data=por_pliegue, x='modelo', y='roc_auc')
plt.title('ROC-AUC por pliegue')
plt.show()

In [ ]:
brecha = (resultados['train_roc_auc_mean'] - resultados['roc_auc_mean']).sort_values(ascending=False)
brecha.round(3)

**Resultado:** `hgb` tiene el mejor ROC-AUC (0.887 ± 0.019), seguido de `forest` (0.866) y `logreg` (0.858). La brecha entrenamiento − validación es mayor en los modelos más flexibles: `forest` (0.133) y `hgb` (0.107) sobreajustan (casi 1.0 en entrenamiento), mientras que `logreg` casi no tiene brecha (0.006) pero rinde menos.

### Paso 50. Finalistas

La elección de finalistas (hgb, forest y logreg) y su justificación están en `reports/problema_ml.md`, sección "Selección de modelos".

## 13. Evaluación en profundidad (solo con entrenamiento)

### Paso 51. Predicciones fuera de muestra y matriz de confusión

In [ ]:
finalistas = {n: ml.build_pipeline(ml.get_candidates(42)[n]) for n in ['logreg', 'forest', 'hgb']}
proba = {n: cross_val_predict(p, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
         for n, p in finalistas.items()}
pred = (proba['hgb'] >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(y_train, pred)
plt.title('Matriz de confusión (hgb, umbral 0.5)')
plt.show()
print(classification_report(y_train, pred))

**Resultado:** con umbral 0.5, `hgb` tiene exactitud 0.82, precisión 0.79 y recall 0.72 para los sobrevivientes. En términos del seguro: verdadero positivo = se predice que sobrevive y sobrevivió; falso positivo = se predice que sobrevive pero murió (se subestima el riesgo); falso negativo = se predice que muere pero sobrevivió (se sobreestima el riesgo); verdadero negativo = se predice que muere y murió.

### Paso 52. Curvas ROC y de precisión-recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for nombre, p in proba.items():
    RocCurveDisplay.from_predictions(y_train, p, name=nombre, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_train, p, name=nombre, ax=axes[1])
axes[0].set_title('Curva ROC')
axes[1].set_title('Curva precisión-recall')
plt.tight_layout()
plt.show()

**Resultado:** el orden de las curvas coincide con el ROC-AUC de la tabla (`hgb` > `forest` > `logreg`). La curva de precisión-recall es más informativa que la ROC cuando las clases están desbalanceadas, porque se centra en la clase positiva; aquí el desbalance es moderado (38 % de positivos).

### Paso 53. Calibración y umbral

In [ ]:
prob_true, prob_pred = calibration_curve(y_train, proba['hgb'], n_bins=8)
plt.plot(prob_pred, prob_true, marker='o', label='hgb')
plt.plot([0, 1], [0, 1], '--', label='calibración perfecta')
plt.xlabel('Probabilidad predicha')
plt.ylabel('Proporción real de sobrevivientes')
plt.title('Curva de calibración')
plt.legend()
plt.show()
print('Brier score:', round(brier_score_loss(y_train, proba['hgb']), 4))

In [ ]:
prec, rec, umbrales = precision_recall_curve(y_train, proba['hgb'])
f1 = 2 * prec * rec / (prec + rec)
mejor = np.nanargmax(f1[:-1])
print('Umbral que maximiza F1:', round(umbrales[mejor], 3), '-> F1 =', round(f1[mejor], 4))
print('F1 con umbral 0.5:', round(f1_score(y_train, proba['hgb'] >= 0.5), 4))

**Resultado:** el Brier score del `hgb` es 0.130, lejos de 0.25 (azar) y de 0 (perfecto). Con este modelo (parámetros por defecto) el umbral que maximiza F1 es 0.61, pero F1 casi no cambia respecto a 0.5 (0.759 vs. 0.755). El umbral final se decide después del ajuste (Paso 58).

### Paso 54. Errores por subgrupos

In [ ]:
errores = X_train.copy()
errores['y'] = y_train
errores['proba'] = proba['hgb']
errores['pred'] = pred
errores['error'] = errores['pred'] != errores['y']
errores['Person'] = np.where(errores['Age'] < 16, 'child', errores['Sex'])
for grupo in ['Sex', 'Pclass', 'Person']:
    tabla = errores.groupby(grupo).apply(
        lambda g: pd.Series({'n': len(g), 'exactitud': 1 - g['error'].mean(),
                             'recall': g.loc[g['y'] == 1, 'pred'].mean()}),
        include_groups=False)
    display(tabla.round(3))

In [ ]:
errores['confianza_error'] = np.where(errores['error'], (errores['proba'] - 0.5).abs(), -1)
errores.sort_values('confianza_error', ascending=False).head(10)[
    ['Sex', 'Pclass', 'Age', 'Fare', 'TTitulo', 'Alone', 'y', 'proba']].round(2)

**Resultado:** la exactitud es parecida entre hombres (82.8 %) y mujeres (80.6 %), pero el recall es muy distinto: 35 % en hombres (n = 459) y 89 % en mujeres (n = 253); en tercera clase el recall es 49.5 % (n = 391). El modelo falla sobre todo con los hombres y los pasajeros de tercera clase que sí sobrevivieron. Los errores de alta confianza son hombres de segunda y tercera clase que sobrevivieron con probabilidad casi 0, y mujeres de primera y segunda clase que murieron con probabilidad ≈ 0.99: casos que contradicen el patrón de sexo y clase. Con las variables actuales no hay forma de distinguirlos.

### Paso 55. Curva de aprendizaje

In [ ]:
tamanos, entren, valid = learning_curve(finalistas['hgb'], X_train, y_train, cv=cv,
                                        train_sizes=np.linspace(0.2, 1.0, 5), scoring='roc_auc')
for nombre, valores in [('entrenamiento', entren), ('validación', valid)]:
    plt.plot(tamanos, valores.mean(axis=1), marker='o', label=nombre)
    plt.fill_between(tamanos, valores.mean(axis=1) - valores.std(axis=1),
                     valores.mean(axis=1) + valores.std(axis=1), alpha=0.2)
plt.xlabel('Pasajeros de entrenamiento')
plt.ylabel('ROC-AUC')
plt.title('Curva de aprendizaje (hgb)')
plt.legend()
plt.show()

**Resultado:** la ROC-AUC de validación sube de 0.80 (113 pasajeros) a 0.89 (569), mientras la de entrenamiento está cerca de 0.99. Hay una brecha grande que se reduce con más datos: señal de **alta varianza**, así que más datos o más regularización ayudarían.

## 14. Ajuste de hiperparámetros

### Paso 57. Búsqueda con validación cruzada (solo entrenamiento)

In [ ]:
grids = ml.get_param_grids()
busquedas = {}
for nombre in ['logreg', 'forest', 'hgb']:
    gs = GridSearchCV(ml.build_pipeline(ml.get_candidates(42)[nombre]), grids[nombre],
                      scoring=ml.PRIMARY_METRIC, cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    busquedas[nombre] = gs
    print(nombre, gs.best_params_, round(gs.best_score_, 4))
    top = pd.DataFrame(gs.cv_results_).sort_values('rank_test_score').head(5)
    display(top[['mean_test_score', 'std_test_score', 'params']].round(4))

**Resultado:** el mejor `logreg` es C = 1 (0.8585), el mejor `forest` 0.8822 y el mejor `hgb` 0.8925 (profundidad 4). Los 5 mejores `hgb` quedan entre 0.8899 y 0.8925, así que la superficie es plana y se prefiere el más simple: profundidad 2 (0.8910), que es lo que queda en `FINAL_PARAMS`.

### Paso 58. ¿Mejoró de verdad? (comparación pareada, mismos pliegues)

In [ ]:
for nombre, gs in busquedas.items():
    base = cross_val_score(ml.build_pipeline(ml.get_candidates(42)[nombre]), X_train, y_train,
                           cv=cv, scoring='roc_auc')
    ajustado = cross_val_score(gs.best_estimator_, X_train, y_train, cv=cv, scoring='roc_auc')
    dif = ajustado - base
    print(nombre, 'diferencia media:', round(dif.mean(), 4), '| desviación:', round(dif.std(), 4))

**Resultado:** la mejora del ajuste es 0 para `logreg`, +0.0165 (± 0.0141) para `forest` y +0.0057 (± 0.0103) para `hgb`. En `hgb` la mejora es menor que la variabilidad entre pliegues, así que el ajuste aportó poco; además `best_score_` es optimista. En `forest` la mejora es del mismo orden que la variabilidad.

**Modelo elegido:** `hgb` con `max_depth=2`, `learning_rate=0.1`, `max_iter=200` y `l2_regularization=1.0` (están en `FINAL_PARAMS`). Con ellos se vuelve a calcular el umbral:

In [ ]:
elegido = ml.build_pipeline(ml.get_candidates(42)['hgb']).set_params(**ml.FINAL_PARAMS)
proba_elegido = cross_val_predict(elegido, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
prec, rec, umbrales = precision_recall_curve(y_train, proba_elegido)
f1 = 2 * prec * rec / (prec + rec)
mejor = np.nanargmax(f1[:-1])
print('Umbral que maximiza F1:', round(umbrales[mejor], 3), '-> F1 =', round(f1[mejor], 4))
print('F1 con umbral 0.5:', round(f1_score(y_train, proba_elegido >= 0.5), 4))
print('THRESHOLD en el módulo:', ml.THRESHOLD)

**Decisión:** con el modelo elegido el umbral que maximiza F1 fuera de muestra es 0.426 (F1 0.782 frente a 0.758 con 0.5). Se fija `THRESHOLD = 0.43`, que favorece un poco el recall. La mejora es modesta y el umbral depende del modelo (con el `hgb` por defecto salía 0.61), así que no hay que darle demasiada precisión.

## 15. Interpretación y ablación

### Paso 59. Coeficientes de la regresión logística

In [ ]:
pipe_lr = ml.build_pipeline(ml.get_candidates(42)['logreg']).fit(X_train, y_train)
coef = pd.DataFrame({'variable': pipe_lr['prep'].get_feature_names_out(),
                     'coef': pipe_lr['model'].coef_[0]})
coef['odds_ratio'] = np.exp(coef['coef'])
coef.reindex(coef['coef'].abs().sort_values(ascending=False).index).round(3)

**Resultado (las numéricas están estandarizadas):** (1) ser de primera clase multiplica los momios de sobrevivir por 3.2 frente al promedio de las clases; (2) ser mujer los multiplica por 2.55; (3) un aumento de una desviación estándar en la edad (≈ 14 años) reduce los momios a 0.52 veces, es decir, casi a la mitad.

### Paso 60. Importancia por permutación

In [ ]:
X_tr, X_val, y_tr, y_val = ml.split(X_train, y_train, test_size=0.25)
modelo = ml.build_pipeline(ml.get_candidates(42)['hgb']).set_params(**ml.FINAL_PARAMS)
modelo.fit(X_tr, y_tr)
r = permutation_importance(modelo, X_val, y_val, scoring='roc_auc', n_repeats=30, random_state=42)
imp = pd.DataFrame({'variable': X_val.columns, 'media': r.importances_mean,
                    'desv': r.importances_std}).sort_values('media', ascending=False)
plt.barh(imp['variable'], imp['media'], xerr=imp['desv'])
plt.gca().invert_yaxis()
plt.xlabel('Caída del ROC-AUC al barajar la variable')
plt.title('Importancia por permutación')
plt.show()
imp.round(4)

**Resultado:** las variables más importantes son `Sex` (cae 0.126 de ROC-AUC al barajarla), `Age` (0.070), `Pclass` (0.063) y `Fare` (0.035); `Parch`, `Alone` y las de cabina no aportan (las de cabina ni siquiera están en el modelo base). El orden coincide con la correlación del Paso 27 en `Sex` y `Pclass`, pero `Age` sube al segundo lugar aunque su correlación lineal con `Survived` es baja (-0.08): el modelo usa la edad en interacción con el sexo y la clase (por ejemplo, los niños), algo que la correlación lineal no ve.

### Paso 61. Ablación de variables (mismos pliegues, hgb ajustado)

In [ ]:
base_n, base_c = ml.NUM_FEATURES, ml.CAT_FEATURES
configs = {
    'A) base': (base_n, base_c),
    'B) base + cabina': (base_n + ml.CABIN_NUM, base_c + ml.CABIN_CAT),
    'C) base sin TTitulo': (base_n, [c for c in base_c if c != 'TTitulo']),
    'D) base sin Sex': (base_n, [c for c in base_c if c != 'Sex']),
}
filas = []
for nombre, (num, cat) in configs.items():
    p = ml.build_pipeline(ml.get_candidates(42)['hgb'], num, cat).set_params(**ml.FINAL_PARAMS)
    s = cross_val_score(p, X_train, y_train, cv=cv, scoring='roc_auc')
    filas.append({'configuración': nombre, 'auc_media': s.mean(), 'auc_desv': s.std()})
ablacion = pd.DataFrame(filas)
ablacion['delta_vs_A'] = ablacion['auc_media'] - ablacion.loc[0, 'auc_media']
ablacion.round(4)

**Resultado:** base 0.8910; con cabina 0.8872 (-0.0038); sin `TTitulo` 0.8875 (-0.0035); sin `Sex` 0.8812 (-0.0098). **Decisión: no incluir las variables de cabina** (`USE_CABIN_FEATURES = False`): no mejoran el desempeño (el cambio es mucho menor que la desviación entre pliegues, ≈ 0.026) y además pueden ser un artefacto del registro (la cabina se conservó más para quienes sobrevivieron). Sin `Sex` el AUC baja poco porque `TTitulo` (Mr, Mrs, Miss) y otras variables toman el relevo.

## 16. Evaluación final sobre el conjunto de prueba (una sola vez)

### Paso 62. Evaluar en prueba con incertidumbre

In [ ]:
pipe_final = ml.train_final(X_train, y_train)
metricas = ml.evaluate_on_test(pipe_final, X_test, y_test, ml.THRESHOLD)
metricas

In [ ]:
ic = ml.bootstrap_ci(pipe_final, X_test, y_test, ml.THRESHOLD, n_boot=1000)
pd.DataFrame(ic, index=['2.5 %', '97.5 %']).round(3)

**Resultado:** en prueba (n = 179) la exactitud es 0.793 (IC 95 % 0.732–0.849), el F1 0.726 (0.627–0.803) y el ROC-AUC 0.815 (0.740–0.880). Los intervalos son anchos (≈ 0.12 de exactitud y 0.14 de AUC) porque hay pocos pasajeros de prueba. Esto implica que diferencias pequeñas entre modelos no se pueden distinguir con este conjunto de prueba.

### Paso 63. Comparación final

In [ ]:
def metricas_binarias(real, pred, score):
    return {'accuracy': accuracy_score(real, pred), 'f1': f1_score(real, pred),
            'roc_auc': roc_auc_score(real, score)}


# Validación cruzada (entrenamiento) y prueba para cada alternativa
dummy_cv = cross_validate(dummy, X_train, y_train, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'])
modelo_cv = cross_validate(elegido, X_train, y_train, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'])
pred_regla = (X_test['Sex'] == 'female').astype(int)
dummy_test = ml.evaluate_on_test(dummy.fit(X_train, y_train), X_test, y_test, 0.5)

columnas = ['accuracy', 'f1', 'roc_auc']
tabla_cv = pd.DataFrame({
    'regla de género': regla[columnas],
    'dummy': {m: dummy_cv[f'test_{m}'].mean() for m in columnas},
    'hgb elegido': {m: modelo_cv[f'test_{m}'].mean() for m in columnas},
}).T.add_suffix(' (CV)')
tabla_prueba = pd.DataFrame({
    'regla de género': metricas_binarias(y_test, pred_regla, pred_regla),
    'dummy': {m: dummy_test[m] for m in columnas},
    'hgb elegido': {m: metricas[m] for m in columnas},
}).T.add_suffix(' (prueba)')
tabla_cv.join(tabla_prueba).round(3)

**Resultado:** en validación cruzada el modelo elegido supera a la regla de género (exactitud 0.826 vs. 0.789). En prueba la ventaja es menor (0.793 vs. 0.777) y está dentro del intervalo de incertidumbre. El ROC-AUC de prueba (0.815) es menor que el de validación cruzada (≈ 0.89). Posibles razones: el conjunto de prueba es pequeño y el intervalo es ancho, y la validación cruzada es algo optimista porque con ella se eligieron el modelo, los hiperparámetros y el umbral. Se documenta el resultado tal cual, sin volver a ajustar con el conjunto de prueba.

## 17. Modelo guardado

### Paso 69. Prueba de humo con pasajeros sin etiqueta

In [ ]:
pipe, meta = ml.load_model(MODELS / 'titanic_model_v1.joblib')
nuevos = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
print(nuevos.shape)

Se aplican las transformaciones **deterministas** de Silver (`Familia`, `Alone`, título y cabina). `build_silver` no sirve aquí: exige `Survived`, que estos pasajeros no tienen, y sus imputaciones usarían estadísticas de los pasajeros nuevos. La edad y los demás faltantes los imputa el propio pipeline guardado, con lo aprendido en entrenamiento.

In [ ]:
nuevos = (nuevos.pipe(de.add_cabin_features)
                .pipe(de.add_family_features)
                .pipe(de.add_title)
                .pipe(de.consolidate_titles))
X_nuevos = nuevos[ml.CANDIDATE_COLUMNS].copy()
resultado = ml.predict(pipe, X_nuevos, threshold=meta['threshold'])
print('NaN en las predicciones:', resultado.isna().any().any())
print('Títulos nuevos:', set(X_nuevos['TTitulo']) - set(X['TTitulo']))
print('Proporción predicha como sobreviviente:', round(resultado['pred'].mean(), 3))
resultado['proba_survive'].describe()

**Resultado:** el modelo guardado predice sobre los 418 pasajeros sin etiqueta sin generar `NaN` ni encontrar títulos nuevos. Predice que el 36.6 % sobrevive y la probabilidad media es 0.39, parecida a la tasa de supervivencia de entrenamiento (38 %), así que el flujo carga → preprocesa → predice funciona de punta a punta.